# Diffusion Transformers & Text-to-Video

Companion notebook for the [DiT & text-to-video lesson](https://ml-viz-ruby.vercel.app/courses/generative-models/08-diffusion-transformers-and-video).

**The idea in one sentence.** Replace the diffusion **U-Net** with a **Transformer** over latent **patches** (DiT); to make video, generalize patches into **spacetime patches** and **factorize** attention into spatial + temporal to survive the token explosion.

We patchify a latent from scratch, run one DiT-style block, then compute the FLOPs that force video models to factorize attention — the quantitative heart of the lesson.

> **To save your work:** click **Copy to Drive** at the top, or File -> Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## 1. Patchify a latent, from scratch

DiT runs in a VAE **latent** space. The denoiser first cuts the latent grid into patches and flattens each into a token — exactly like a Vision Transformer. Here is a synthetic latent `(C, H, W)` and a `patchify` that returns a `(num_patches, patch_dim)` token matrix.

In [ ]:
C, H, W, P = 4, 8, 8, 2          # channels, height, width, patch size
latent = np.random.randn(C, H, W)

def patchify(x, p):
    C, H, W = x.shape
    tokens = []
    for i in range(0, H, p):
        for j in range(0, W, p):
            tokens.append(x[:, i:i+p, j:j+p].reshape(-1))
    return np.stack(tokens)

tokens = patchify(latent, P)
print('latent', latent.shape, '-> tokens', tokens.shape,
      '  (' + str((H//P)*(W//P)) + ' patches x ' + str(C*P*P) + ' dims)')

## 2. One DiT block: attention over patches + adaLN conditioning

A DiT block is standard self-attention plus **adaptive layer norm (adaLN)**: the conditioning vector (diffusion timestep `t` + prompt) predicts a per-block **scale** and **shift**. We implement a minimal, un-trained forward pass to see the shapes and the conditioning path.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def attention(X, Wq, Wk, Wv):
    q, k, v = X @ Wq, X @ Wk, X @ Wv
    d = q.shape[-1]
    return softmax(q @ k.T / np.sqrt(d)) @ v

def ada_ln(X, scale, shift):
    mu, sd = X.mean(-1, keepdims=True), X.std(-1, keepdims=True) + 1e-5
    return (X - mu) / sd * (1 + scale) + shift

N, D = tokens.shape
Wq = np.random.randn(D, D) * 0.02
Wk = np.random.randn(D, D) * 0.02
Wv = np.random.randn(D, D) * 0.02

cond = np.random.randn(D) * 0.02          # timestep + prompt embedding
scale, shift = cond, cond                  # a real DiT uses small MLPs here

h = ada_ln(tokens, scale, shift)
h = tokens + attention(h, Wq, Wk, Wv)      # residual
noise_pred = h                             # a linear head would un-patchify this
print('predicted-noise tokens:', noise_pred.shape, '(one vector per patch)')

**What to notice.** Nothing here is convolutional — every patch can attend to every other patch in one layer (global context). The timestep/prompt enter *only* through adaLN's `scale`/`shift`, which is how DiT stays a clean Transformer while still being conditioned on `t` and the text prompt.

## 3. The library way

Real DiTs are in `diffusers`. The image pipeline is a few lines (weights download on first run):

In [ ]:
sketch = '''
from diffusers import DiTPipeline, DPMSolverMultistepScheduler
import torch

pipe = DiTPipeline.from_pretrained('facebook/DiT-XL-2-256', torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
image = pipe(class_labels=[207], num_inference_steps=25).images[0]  # 207 = golden retriever
'''
print(sketch)
print('For video: hpcaitech/Open-Sora wraps a spatial-temporal DiT (STDiT) + a 3D VAE.')

## 4. Why video must factorize attention

A video latent has `T` frames of an `H x W` grid -> `N = T * (H/P) * (W/P)` spacetime tokens. Full self-attention is `O(N^2)`. **Factorized** attention runs spatial attention within each frame plus temporal attention across frames, costing `O(T * S^2) + O(S * T^2)` where `S` is tokens per frame. Let us compute how those diverge as frames grow.

In [ ]:
S = (H//P) * (W//P)                  # spatial tokens per frame
frames = np.arange(1, 65)
full = (frames * S)**2
factorized = frames * S**2 + S * frames**2
for T in [4, 16, 64]:
    f = (T*S)**2; fac = T*S**2 + S*T**2
    print('T=' + str(T).rjust(2), ' full/factorized attention cost ratio =', round(f/fac, 1), 'x')

### Visualize it

In [ ]:
plt.figure()
plt.plot(frames, full, label='full 3D attention  O(N^2)')
plt.plot(frames, factorized, label='factorized (spatial + temporal)')
plt.yscale('log')
plt.xlabel('number of frames  T')
plt.ylabel('attention operations (log scale)')
plt.title('Full vs factorized attention as video gets longer')
plt.legend()
plt.show()

**What to notice.** The gap widens fast: full attention grows with the **square of total tokens**, so doubling the frames quadruples cost. Factorization (STDiT, Latte) is what keeps long, high-res video generation tractable — Open-Sora reports up to ~5x speedup as frames grow.

## 5. Tradeoffs & failure modes

| Aspect | U-Net diffusion | **DiT / video DiT** |
|---|---|---|
| Inductive bias | strong (conv locality) | weak (learned) |
| Scaling | plateaus | smooth with compute |
| Video compute | - | very high, O(N^2) |
| Data hunger | lower | higher |

**Failure modes:** temporal flicker / identity drift across frames, implausible motion, and raw cost. Levers: factorized/windowed attention, a **3D VAE** to shrink tokens, and classifier-free guidance for prompt adherence.

## 6. Your turn

Add a **3D VAE** temporal compression factor `tc` that shrinks the frame count *before* patchifying (`T_eff = ceil(T / tc)`). Re-plot the factorized cost for `tc = 1, 2, 4` and confirm temporal compression buys a large, roughly `tc^2`, saving on the temporal term.

In [ ]:
def factorized_cost(T, S, tc=1):
    # TODO(you): compress frames by tc before computing the cost
    raise NotImplementedError

# for tc in [1, 2, 4]:
#     plt.plot(frames, [factorized_cost(T, S, tc) for T in frames], label='tc=' + str(tc))
print('implement factorized_cost, then compare tc = 1, 2, 4')

<details><summary>Solution</summary>

```python
def factorized_cost(T, S, tc=1):
    Te = int(np.ceil(T / tc))
    return Te * S**2 + S * Te**2

plt.figure()
for tc in [1, 2, 4]:
    plt.plot(frames, [factorized_cost(T, S, tc) for T in frames], label='tc=' + str(tc))
plt.yscale('log'); plt.legend(); plt.xlabel('frames T'); plt.show()
```

The temporal term `S * Te^2` shrinks by ~`tc^2`, which is why every serious video model compresses time with a 3D VAE before the transformer sees it.
</details>

## 7. Key takeaways

- **DiT** = Transformer denoiser over latent **patches**, conditioned via **adaLN** — and it scales predictably with compute.
- **Video** = **spacetime patches**; full 3D attention is `O(N^2)` and infeasible, so models **factorize** into spatial + temporal (STDiT).
- A **3D VAE** compresses time to cut tokens before the transformer runs.

Next: [ViT & Modern GenAI](https://ml-viz-ruby.vercel.app/courses/generative-models/06-vit-and-modern-genai) for classifier-free guidance and latent diffusion.